# Tool-call fine-tuning on a free Colab T4: baseline → QLoRA SFT → DPO

Runs the whole pipeline from this repo end to end on a **free Colab T4 (16 GB)**:

1. Build the cleaned glaive-function-calling-v2 splits (3,000 train / 400 val / 600 test, 20% "no tool applies").
2. **Baseline**: base model + engineered few-shot prompt, scored on the held-out test set.
3. **Phase 1 – SFT**: QLoRA (4-bit NF4) with TRL `SFTTrainer` + PEFT.
4. **Phase 2 – DPO**: sample 4 outputs/prompt from the SFT model, auto-label chosen/rejected with the scorer, train with TRL `DPOTrainer` stacked on the SFT adapter.
5. Re-evaluate and plot curves + the before/after table.

**Runtime → Change runtime type → T4 GPU** before running. Default config: `configs/colab_t4_qwen3_8b.yaml` (Qwen3-8B, fp16 compute). It has not been executed by the author on Colab; if you hit OOM or the session time limit, set `CONFIG = "configs/colab_t4_small.yaml"` below (Qwen2.5-0.5B-Instruct, minutes instead of hours).

In [ ]:
# 1) Get the code. Either clone your fork of the repo...
REPO_URL = ""  # e.g. "https://github.com/<you>/04-finetune-sft-dpo.git"
import os, subprocess
if REPO_URL:
    subprocess.run(["git", "clone", REPO_URL, "repo"], check=True)
    os.chdir("repo")
else:
    # ...or upload the repo as a zip via the Colab file browser, then:
    from google.colab import files  # noqa
    print("Upload 04-finetune-sft-dpo.zip")
    up = files.upload()
    name = next(iter(up))
    subprocess.run(["unzip", "-q", "-o", name], check=True)
    os.chdir(name.replace(".zip", ""))
print(os.getcwd())

In [ ]:
# 2) Install (Colab already ships CUDA torch)
!pip -q install "transformers==4.57.1" "trl==0.24.0" "peft==0.17.1" "datasets==4.3.0" "accelerate>=1.10" "bitsandbytes>=0.45" pyyaml matplotlib tensorboard openai
!nvidia-smi --query-gpu=name,memory.total --format=csv

In [ ]:
import os, sys
CONFIG = "configs/colab_t4_qwen3_8b.yaml"   # or "configs/colab_t4_small.yaml" for a quick run
os.environ["PYTHONPATH"] = os.path.abspath("src")
sys.path.insert(0, os.path.abspath("src"))
# Optional: log to Weights & Biases by adding "wandb" to report_to in the YAML and:
# import wandb; wandb.login()

In [ ]:
# 3) Data: processed splits are committed in data/processed; rebuild them from the pinned raw file if you want
# !python -m toolcall_ft.prepare_data --config $CONFIG
!python -c "import json;print(json.dumps(json.load(open('data/processed/stats.json')), indent=1))"

In [ ]:
# 4) Baseline: base model + engineered few-shot prompt on the held-out test set
!python -m toolcall_ft.evaluate --config $CONFIG --stage baseline_fewshot

In [ ]:
# 5) Phase 1: QLoRA SFT (TRL SFTTrainer, loss on the assistant JSON only)
!python -m toolcall_ft.train_sft --config $CONFIG
!python -m toolcall_ft.evaluate --config $CONFIG --stage sft

In [ ]:
# 6) Phase 2: generate 4 samples/prompt from the SFT model, auto-label pairs with the scorer, DPO on top of SFT
!python -m toolcall_ft.build_dpo_pairs --config $CONFIG
!python -m toolcall_ft.train_dpo --config $CONFIG
!python -m toolcall_ft.evaluate --config $CONFIG --stage sft_dpo

In [ ]:
# 7) Report: results table + training curves
!python -m toolcall_ft.report --config $CONFIG
import yaml, glob
from IPython.display import Image, Markdown, display
res = yaml.safe_load(open(CONFIG))["results_dir"]
display(Markdown(open(f"{res}/results_table.md").read()))
for p in sorted(glob.glob(f"{res}/plots/*.png")):
    display(Image(p))

In [ ]:
# 8) (optional) TensorBoard
%load_ext tensorboard
%tensorboard --logdir outputs

In [ ]:
# 9) Save the adapters to Drive (they are git-ignored on purpose)
# from google.colab import drive; drive.mount('/content/drive')
# !cp -r outputs /content/drive/MyDrive/toolcall_ft_outputs